# Notebook 10: Decision Trees
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Understand how a decision tree splits data
- Compute information gain and Gini impurity
- Grow, visualise, and prune a decision tree
- Control tree depth to prevent overfitting
- Extract feature importances

## 1. What Is a Decision Tree?

A decision tree partitions the feature space into axis-aligned regions, each assigned a prediction. Each **internal node** tests one feature; each **leaf** gives a prediction.

```
             Is petal_length < 2.45?
            /                        \
          YES                         NO
       Setosa                 Is petal_width < 1.75?
                              /                    \
                           YES                      NO
                        Versicolor              Virginica
```

**Why trees?**
- Human-readable rules — fully interpretable
- Handle mixed feature types natively
- No feature scaling required
- Naturally capture non-linear interactions

## 2. Splitting Criteria

### Gini Impurity
$$G = 1 - \sum_{k=1}^K p_k^2$$

$G = 0$ means a pure node (only one class). $G = 0.5$ is maximum impurity for a binary problem.

### Information Gain (Entropy)
$$H = -\sum_{k=1}^K p_k \log_2 p_k$$
$$\text{IG} = H(\text{parent}) - \frac{n_L}{n} H(\text{left}) - \frac{n_R}{n} H(\text{right})$$

At each node, the tree exhaustively searches **all features and all thresholds** to find the split with the highest Information Gain (or lowest Gini after split).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# Visualise Gini and Entropy
p = np.linspace(0.001, 0.999, 300)
gini    = 1 - p**2 - (1-p)**2
entropy = -(p*np.log2(p) + (1-p)*np.log2(1-p))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, gini,          label='Gini impurity',     linewidth=2)
ax.plot(p, entropy * 0.5, label='Entropy (×0.5)',    linewidth=2, linestyle='--')
ax.set_xlabel('p (fraction of positive class)')
ax.set_ylabel('Impurity')
ax.set_title('Gini vs Entropy Impurity Measures')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Manual information-gain calculation
def entropy(y):
    _, counts = np.unique(y, return_counts=True)
    p = counts / counts.sum()
    return -np.sum(p * np.log2(p + 1e-12))

def info_gain(y, y_left, y_right):
    n, n_l, n_r = len(y), len(y_left), len(y_right)
    return entropy(y) - (n_l/n)*entropy(y_left) - (n_r/n)*entropy(y_right)

iris = load_iris()
X, y = iris.data, iris.target
petal_length = X[:, 2]

best_t, best_ig = max(
    ((t, info_gain(y, y[petal_length<=t], y[petal_length>t]))
     for t in np.unique(petal_length)
     if len(y[petal_length<=t])>0 and len(y[petal_length>t])>0),
    key=lambda x: x[1]
)
print(f'Best threshold for petal_length: {best_t:.1f} cm')
print(f'Information Gain               : {best_ig:.4f}')

## 3. Growing a Tree with scikit-learn

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_tr, y_tr)

print(f'Train accuracy: {tree.score(X_tr, y_tr):.3f}')
print(f'Test  accuracy: {tree.score(X_te, y_te):.3f}')

fig, ax = plt.subplots(figsize=(16, 6))
plot_tree(tree, feature_names=iris.feature_names,
          class_names=iris.target_names,
          filled=True, ax=ax, fontsize=8)
ax.set_title('Decision Tree (max_depth=3) — Iris')
plt.tight_layout()
plt.show()

print('\nText representation:')
print(export_text(tree, feature_names=list(iris.feature_names)))

## 4. Bias-Variance via Depth

In [ ]:
depths = range(1, 15)
train_accs, test_accs = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_tr, y_tr)
    train_accs.append(t.score(X_tr, y_tr))
    test_accs.append(t.score(X_te, y_te))

best_d = list(depths)[np.argmax(test_accs)]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depths, train_accs, marker='o', label='Train')
ax.plot(depths, test_accs,  marker='s', label='Test')
ax.axvline(best_d, color='red', linestyle='--', label=f'Best depth={best_d}')
ax.set_xlabel('max_depth'); ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Accuracy vs Depth (Iris)')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Unconstrained tree: train={train_accs[-1]:.3f}, test={test_accs[-1]:.3f} — overfitting!')
print(f'Best depth={best_d}: test accuracy={max(test_accs):.3f}')

## 5. Feature Importances

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

tree_c = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_c.fit(X_tr_c, y_tr_c)
print(f'Breast Cancer — Test accuracy: {tree_c.score(X_te_c, y_te_c):.3f}')

imps = tree_c.feature_importances_
idx = np.argsort(imps)[::-1][:10]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(10), imps[idx], color='steelblue')
ax.set_xticks(range(10))
ax.set_xticklabels([cancer.feature_names[i] for i in idx], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Importance')
ax.set_title('Top 10 Feature Importances')
plt.tight_layout()
plt.show()

## 6. Decision Boundary Visualisation

In [ ]:
from matplotlib.colors import ListedColormap

X2 = iris.data[:, 2:4]  # petal length & width
dt2 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X2, iris.target)

x_min, x_max = X2[:,0].min()-0.3, X2[:,0].max()+0.3
y_min, y_max = X2[:,1].min()-0.3, X2[:,1].max()+0.3
xx, yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
Z = dt2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF9999','#99FF99','#9999FF']))
ax.scatter(X2[:,0], X2[:,1], c=iris.target,
           cmap=ListedColormap(['#CC0000','#00AA00','#0000CC']),
           edgecolors='white', s=40)
ax.set_xlabel('Petal Length (cm)'); ax.set_ylabel('Petal Width (cm)')
ax.set_title('Decision Tree Regions (Iris — petal features)')
plt.tight_layout()
plt.show()

## Exercises

1. Train with `max_depth=None`. What is the training accuracy? What does this tell you about overfitting?
2. Use `cost_complexity_pruning_path` to find the optimal `ccp_alpha` via cross-validation.
3. Implement Gini impurity manually and verify it matches `tree.tree_.impurity` for the root node.
4. Apply a decision tree to the Titanic dataset and extract the rules as text — what does the tree say is most predictive of survival?

## Reflection Questions
- Decision trees create axis-aligned splits. What kinds of true decision boundaries are hard to represent?
- Why do trees overfit when unconstrained, and what are three hyperparameters that control complexity?
- Feature importances from a single tree can be noisy. How do Random Forests address this?

---
**Next →** `11_ensemble_methods.ipynb`